<a href="https://colab.research.google.com/github/beast-gif/apsynth/blob/main/DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 !pip install pretty_midi sentence-transformers -q
print("✅ Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 49.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.8 MB/s eta 0:00:00
✅ Done


In [ ]:
import os

# Only mount if not already mounted
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Drive mounted")
else:
    print("✅ Drive already mounted")

# Create all folders
dirs = [
    "/content/music_generator",
    "/content/music_generator/data",
    "/content/music_generator/data/cache",
    "/content/music_generator/model",
    "/content/music_generator/midi_utils",
    "/content/music_generator/ui",
    "/content/drive/MyDrive/music_generator/cache",
    "/content/drive/MyDrive/music_generator/checkpoints/bass",
    "/content/drive/MyDrive/music_generator/checkpoints/chords",
    "/content/drive/MyDrive/music_generator/checkpoints/melody",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

for d in ["/content/music_generator/data",
          "/content/music_generator/model",
          "/content/music_generator/midi_utils"]:
    open(f"{d}/__init__.py", "w").close()

print("✅ All folders ready")

Mounted at /content/drive
✅ Drive mounted
✅ All folders ready


In [ ]:
import os, sys

config_text = """
import os

BASE_DIR    = "/content/music_generator"
DATA_DIR    = os.path.join(BASE_DIR, "data")
CACHE_DIR   = os.path.join(DATA_DIR, "cache")
POP909_DIR  = os.path.join(DATA_DIR, "pop909/POP909-Dataset-master/POP909")
DRIVE_CACHE = "/content/drive/MyDrive/music_generator/cache"
DRIVE_CKPT  = "/content/drive/MyDrive/music_generator/checkpoints"

# ── Smaller vocab = easier to learn = lower loss ──────────
N_VELOCITY_BINS = 4   # was 8
N_TIME_BINS     = 12  # was 16
N_DURATION_BINS = 4   # was 8
N_PITCH_BINS    = 72  # was 88 → C2 to B7, covers 99% of bass notes
MIDI_PITCH_MIN  = 24  # C1
MIDI_PITCH_MAX  = 96  # C7

DRUM_PITCH_MAP = {
    36:0, 38:1, 40:1, 42:2, 44:2, 46:3,
    49:4, 51:5, 53:5, 47:6, 48:6, 45:7, 43:7, 41:8
}
N_DRUM_INSTRUMENTS = 9

DRUM_CONTENT_VOCAB    = N_DRUM_INSTRUMENTS * N_VELOCITY_BINS * N_TIME_BINS
PITCHED_CONTENT_VOCAB = N_PITCH_BINS * N_VELOCITY_BINS * N_TIME_BINS * N_DURATION_BINS
# = 72 × 4 × 12 × 4 = 13,824  ← was 90,112!

N_SPECIALS = 6

DRUM_VOCAB_SIZE    = DRUM_CONTENT_VOCAB    + N_SPECIALS
PITCHED_VOCAB_SIZE = PITCHED_CONTENT_VOCAB + N_SPECIALS

def make_specials(content_size):
    return {
        "BOS":    content_size + 0,
        "EOS":    content_size + 1,
        "PAD":    content_size + 2,
        "UNCOND": content_size + 3,
        "COND":   content_size + 4,
        "MARKER": content_size + 5,
    }

DRUM_SPECIALS    = make_specials(DRUM_CONTENT_VOCAB)
PITCHED_SPECIALS = make_specials(PITCHED_CONTENT_VOCAB)

BOS_TOKEN    = DRUM_SPECIALS["BOS"]
EOS_TOKEN    = DRUM_SPECIALS["EOS"]
PAD_TOKEN    = DRUM_SPECIALS["PAD"]
UNCOND_TOKEN = DRUM_SPECIALS["UNCOND"]
COND_TOKEN   = DRUM_SPECIALS["COND"]
VOCAB_SIZE   = DRUM_VOCAB_SIZE

D_MODEL     = 256
N_HEADS     = 4
N_LAYERS    = 6
D_FF        = 1024
DROPOUT     = 0.2
MAX_SEQ_LEN = 256
CONTEXT_LEN = 256

BATCH_SIZE    = 8
LEARNING_RATE = 5e-4
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = 1000
MAX_STEPS     = 50000
GRAD_CLIP     = 1.0
SAVE_EVERY    = 2000
LOG_EVERY     = 100
EVAL_EVERY    = 500
PATIENCE      = 10

GEN_TEMPERATURE = 1.0
GEN_TOP_P       = 0.92
GEN_MAX_TOKENS  = 512

BASS_PROGRAM   = 32
CHORD_PROGRAM  = 0
MELODY_PROGRAM = 0
"""

with open("/content/music_generator/config.py", "w") as f:
    f.write(config_text)
# Add to existing config
with open("/content/music_generator/config.py", "a") as f:
    f.write("""
# ── Text encoder ──────────────────────────────────────────
TEXT_EMB_DIM = 384    # sentence-transformers MiniLM output dim
TEXT_MODEL   = "all-MiniLM-L6-v2"   # lightweight + fast
""")
print("✅ Config updated with text encoder settings")

for mod in list(sys.modules.keys()):
    if "config" in mod: del sys.modules[mod]
sys.path.insert(0, "/content/music_generator")
import config as C

print(f"✅ New config!")
print(f"   PITCHED_VOCAB:  {C.PITCHED_VOCAB_SIZE:,}  (was 90,118)")
print(f"   Logits RAM:     {8 * 256 * C.PITCHED_VOCAB_SIZE * 4 / 1e6:.0f} MB")
print(f"   Embedding RAM:  {C.PITCHED_VOCAB_SIZE * 256 * 4 / 1e6:.0f} MB")

✅ Config updated with text encoder settings
✅ New config!
   PITCHED_VOCAB:  13,830  (was 90,118)
   Logits RAM:     113 MB
   Embedding RAM:  14 MB


In [ ]:
tok_code = '''
import pretty_midi, sys
sys.path.insert(0, "/content/music_generator")
import config as C

class UnifiedTokenizer:
    def __init__(self, track="bass"):
        self.track   = track
        self.n_vel   = C.N_VELOCITY_BINS
        self.n_time  = C.N_TIME_BINS
        self.n_dur   = C.N_DURATION_BINS
        self.n_pitch = C.N_PITCH_BINS

        if track == "drums":
            sp = C.DRUM_SPECIALS
            self.content_vocab = C.DRUM_CONTENT_VOCAB
            self.vocab_size    = C.DRUM_VOCAB_SIZE
        else:
            sp = C.PITCHED_SPECIALS
            self.content_vocab = C.PITCHED_CONTENT_VOCAB
            self.vocab_size    = C.PITCHED_VOCAB_SIZE

        self.BOS    = sp["BOS"]
        self.EOS    = sp["EOS"]
        self.PAD    = sp["PAD"]
        self.UNCOND = sp["UNCOND"]
        self.COND   = sp["COND"]
        self.MARKER = sp["MARKER"]
        self.SPECIALS = {self.BOS, self.EOS, self.PAD,
                         self.UNCOND, self.COND, self.MARKER}

    def get_tempo(self, pm):
        try:
            _, vals = pm.get_tempo_change_times()
            return float(vals[0]) if len(vals) > 0 else 120.0
        except:
            return 120.0

    def encode_pitched(self, pitch, velocity, time_offset, duration, tempo):
        pitch_bin = min(max(pitch - C.MIDI_PITCH_MIN, 0), self.n_pitch-1)
        vel_bin   = min(int(velocity/128 * self.n_vel), self.n_vel-1)
        beat_dur  = 60.0 / max(tempo, 1.0)
        step_dur  = beat_dur / self.n_time
        time_bin  = min(int(round(time_offset/max(step_dur,1e-9))), self.n_time-1)
        time_bin  = max(0, time_bin)
        dur_bin   = min(int(duration/max(step_dur,1e-9)/(self.n_time/self.n_dur)),
                        self.n_dur-1)
        dur_bin   = max(0, dur_bin)
        return (pitch_bin * self.n_vel * self.n_time * self.n_dur +
                vel_bin   * self.n_time * self.n_dur +
                time_bin  * self.n_dur  + dur_bin)

    def decode_pitched(self, token):
        if token >= self.content_vocab or token < 0:
            return None
        local     = token
        dur_bin   = local % self.n_dur;  local //= self.n_dur
        time_bin  = local % self.n_time; local //= self.n_time
        vel_bin   = local % self.n_vel
        pitch_bin = local // self.n_vel
        return pitch_bin + C.MIDI_PITCH_MIN, vel_bin, time_bin, dur_bin

    def track_to_tokens(self, pm, track_idx, tempo):
        if track_idx >= len(pm.instruments):
            return []
        inst     = pm.instruments[track_idx]
        beat_dur = 60.0 / max(tempo, 1.0)
        events   = []
        for note in inst.notes:
            if not (C.MIDI_PITCH_MIN <= note.pitch <= C.MIDI_PITCH_MAX):
                continue
            beat_num     = int(note.start / beat_dur)
            time_in_beat = note.start - beat_num * beat_dur
            token        = self.encode_pitched(
                note.pitch, note.velocity,
                time_in_beat, note.end - note.start, tempo)
            events.append((note.start, token))
        events.sort(key=lambda x: x[0])
        return [t for _, t in events]

    def pop909_to_tokens(self, midi_path):
        try:
            pm = pretty_midi.PrettyMIDI(midi_path)
        except:
            return None
        tempo = self.get_tempo(pm)
        return {
            "melody": self.track_to_tokens(pm, 0, tempo),
            "chords": self.track_to_tokens(pm, 1, tempo),
            "bass":   self.track_to_tokens(pm, 2, tempo),
            "tempo":  tempo,
        }

    def build_sequence(self, tokens):
        if not tokens:
            return []
        return [self.BOS, self.UNCOND, self.MARKER] + tokens + [self.EOS]

    def chunk_sequence(self, tokens, chunk_len=512, stride=64):
        chunks = []
        for i in range(0, max(1, len(tokens)-chunk_len+1), stride):
            chunk = tokens[i:i+chunk_len]
            if len(chunk) < chunk_len:
                chunk = chunk + [self.PAD]*(chunk_len-len(chunk))
            chunks.append(chunk)
        return chunks

    def tokens_to_midi(self, tokens, program, tempo=120.0):
        pm       = pretty_midi.PrettyMIDI(initial_tempo=tempo)
        inst     = pretty_midi.Instrument(program=program, is_drum=False)
        beat_dur = 60.0 / max(tempo, 1.0)
        step_dur = beat_dur / self.n_time
        WRAP     = self.n_time // 2
        current_beat, prev_tb = 0, -1
        for tok in tokens:
            if tok in self.SPECIALS:
                continue
            decoded = self.decode_pitched(tok)
            if decoded is None:
                continue
            pitch, vel_bin, tb, dur_bin = decoded
            if prev_tb >= 0 and tb < prev_tb - WRAP:
                current_beat += 1
            prev_tb  = tb
            start    = current_beat * beat_dur + tb * step_dur
            duration = max(0.05,(dur_bin+1)*step_dur*(self.n_time/self.n_dur))
            velocity = max(1, min(127, int((vel_bin+0.5)/self.n_vel*127)))
            inst.notes.append(pretty_midi.Note(
                velocity=velocity, pitch=pitch,
                start=start, end=start+duration))
        inst.notes.sort(key=lambda n: n.start)
        pm.instruments.append(inst)
        return pm
'''
with open("/content/music_generator/data/tokenizer.py", "w") as f:
    f.write(tok_code)
print("✅ tokenizer.py written")

✅ tokenizer.py written


In [ ]:
model_code = '''
import math, torch
import torch.nn as nn
import torch.nn.functional as F
import sys
sys.path.insert(0, "/content/music_generator")
import config as C

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, max_len):
        super().__init__()
        self.n_heads    = n_heads
        self.d_head     = d_model // n_heads
        self.scale      = self.d_head ** -0.5
        self.qkv_proj   = nn.Linear(d_model, 3*d_model, bias=False)
        self.out_proj   = nn.Linear(d_model, d_model,   bias=False)
        self.attn_drop  = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.tril(torch.ones(max_len, max_len)).unsqueeze(0).unsqueeze(0))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        attn = (q @ k.transpose(-2,-1)) * self.scale
        attn = attn.masked_fill(self.mask[:,:,:T,:T]==0, float("-inf"))
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        out  = (attn @ v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_drop(self.out_proj(out))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model, bias=False), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, max_len):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout, max_len)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = FeedForward(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class MusicTransformer(nn.Module):
    def __init__(self,
        vocab_size=C.VOCAB_SIZE, d_model=C.D_MODEL,
        n_heads=C.N_HEADS, n_layers=C.N_LAYERS,
        d_ff=C.D_FF, dropout=C.DROPOUT, max_len=C.MAX_SEQ_LEN):
        super().__init__()
        self.vocab_size = vocab_size
        self.token_emb  = nn.Embedding(vocab_size, d_model)
        self.pos_emb    = nn.Embedding(max_len, d_model)
        self.drop       = nn.Dropout(dropout)
        self.blocks     = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout, max_len)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_emb.weight
        self._init_weights()
        n = sum(p.numel() for p in self.parameters())
        print(f"[MusicTransformer] {n/1e6:.2f}M parameters")

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0, 0.02)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, input_ids, labels=None):
        B, T  = input_ids.shape
        pos   = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x     = self.drop(self.token_emb(input_ids) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))
        loss   = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.vocab_size),
                labels.view(-1), ignore_index=-100)
        return logits, loss

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())
'''
with open("/content/music_generator/model/transformer.py", "w") as f:
    f.write(model_code)
print("✅ transformer.py written")

✅ transformer.py written


In [ ]:
import requests, zipfile, os
from tqdm import tqdm

URL      = "https://github.com/music-x-lab/POP909-Dataset/archive/refs/heads/master.zip"
ZIP_PATH = "/content/pop909.zip"
SAVE_DIR = "/content/music_generator/data/pop909"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Downloading POP909...")
r = requests.get(URL, stream=True)
total = int(r.headers.get("content-length", 0))
with open(ZIP_PATH, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
    for chunk in r.iter_content(chunk_size=64*1024):
        f.write(chunk)
        bar.update(len(chunk))

print("Unzipping...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(SAVE_DIR)
os.remove(ZIP_PATH)

import glob
midis = glob.glob(f"{SAVE_DIR}/**/*.mid", recursive=True)
print(f"✅ Found {len(midis)} MIDI files")

49.5MB [00:02, 23.5MB/s]


Unzipping...
✅ Found 2898 MIDI files


In [ ]:
import os, glob, numpy as np, sys
from tqdm import tqdm
import shutil

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["config","tokenizer"]):
        del sys.modules[mod]

sys.path.insert(0, "/content/music_generator")
import config as C

# Inline tokenizer with new vocab
import pretty_midi

def get_tempo(pm):
    try:
        _, vals = pm.get_tempo_change_times()
        return float(vals[0]) if len(vals) > 0 else 120.0
    except:
        return 120.0

def encode_pitched(pitch, velocity, time_offset, duration, tempo):
    pitch_bin = min(max(pitch - C.MIDI_PITCH_MIN, 0), C.N_PITCH_BINS-1)
    vel_bin   = min(int(velocity/128 * C.N_VELOCITY_BINS), C.N_VELOCITY_BINS-1)
    beat_dur  = 60.0 / max(tempo, 1.0)
    step_dur  = beat_dur / C.N_TIME_BINS
    time_bin  = min(int(round(time_offset/max(step_dur,1e-9))), C.N_TIME_BINS-1)
    time_bin  = max(0, time_bin)
    dur_bin   = min(int(duration/max(step_dur,1e-9)/(C.N_TIME_BINS/C.N_DURATION_BINS)),
                    C.N_DURATION_BINS-1)
    dur_bin   = max(0, dur_bin)
    return (pitch_bin * C.N_VELOCITY_BINS * C.N_TIME_BINS * C.N_DURATION_BINS +
            vel_bin   * C.N_TIME_BINS * C.N_DURATION_BINS +
            time_bin  * C.N_DURATION_BINS + dur_bin)

def track_to_tokens(pm, track_idx, tempo):
    if track_idx >= len(pm.instruments):
        return []
    inst     = pm.instruments[track_idx]
    beat_dur = 60.0 / max(tempo, 1.0)
    events   = []
    for note in inst.notes:
        if not (C.MIDI_PITCH_MIN <= note.pitch <= C.MIDI_PITCH_MAX):
            continue
        beat_num     = int(note.start / beat_dur)
        time_in_beat = note.start - beat_num * beat_dur
        token        = encode_pitched(
            note.pitch, note.velocity,
            time_in_beat, note.end - note.start, tempo)
        events.append((note.start, token))
    events.sort(key=lambda x: x[0])
    return [t for _, t in events]

def build_sequence(tokens, sp):
    if not tokens:
        return []
    return [sp["BOS"], sp["UNCOND"], sp["MARKER"]] + tokens + [sp["EOS"]]

def chunk_sequence(tokens, chunk_len, stride):
    chunks = []
    for i in range(0, max(1, len(tokens)-chunk_len+1), stride):
        chunk = tokens[i:i+chunk_len]
        if len(chunk) < chunk_len:
            chunk = chunk + [C.PITCHED_SPECIALS["PAD"]]*(chunk_len-len(chunk))
        chunks.append(chunk)
    return chunks

files = glob.glob(f"{C.POP909_DIR}/**/*.mid", recursive=True)
print(f"Reprocessing {len(files)} files with new vocab...")

TRACK_MAP = {"melody": 0, "chords": 1, "bass": 2}
results   = {"melody": [], "chords": [], "bass": []}

for path in tqdm(files):
    try:
        pm = pretty_midi.PrettyMIDI(path)
    except:
        continue
    tempo = get_tempo(pm)
    sp    = C.PITCHED_SPECIALS

    for track_name, track_idx in TRACK_MAP.items():
        tokens = track_to_tokens(pm, track_idx, tempo)
        seq    = build_sequence(tokens, sp)
        if len(seq) < 16:
            continue
        chunks = chunk_sequence(seq, C.MAX_SEQ_LEN, stride=32)
        results[track_name].extend(chunks)

os.makedirs(C.CACHE_DIR, exist_ok=True)
os.makedirs(C.DRIVE_CACHE, exist_ok=True)

for name, chunks in results.items():
    arr   = np.array(chunks, dtype=np.int32)
    local = f"{C.CACHE_DIR}/{name}_chunks_v2.npy"
    drive = f"{C.DRIVE_CACHE}/{name}_chunks_v2.npy"
    np.save(local, arr)
    shutil.copy(local, drive)
    print(f"✅ {name}: {len(chunks):,} chunks | "
          f"{arr.nbytes/1e6:.1f} MB | max_token={arr.max()}")

print(f"\n✅ Done! New vocab_size = {C.PITCHED_VOCAB_SIZE:,}")

Reprocessing 2898 files with new vocab...


 22%|██▏       | 644/2898 [00:34<02:41, 13.94it/s]/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
100%|██████████| 2898/2898 [02:37<00:00, 18.36it/s]


✅ melody: 10,118 chunks | 10.4 MB | max_token=13829
✅ chords: 8,021 chunks | 8.2 MB | max_token=13829
✅ bass: 73,414 chunks | 75.2 MB | max_token=13829

✅ Done! New vocab_size = 13,830


In [ ]:
import os, math, time, torch, torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
import sys
sys.path.insert(0, "/content/music_generator")
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["config","transformer"]):
        del sys.modules[mod]
import config as C
from model.transformer import MusicTransformer
import gc
torch.cuda.empty_cache(); gc.collect()

# ── CHANGE THIS ONLY ──
TRACK = "bass"
# ──────────────────────

VOCAB_SIZE = C.PITCHED_VOCAB_SIZE
PAD_ID     = C.PITCHED_SPECIALS["PAD"]
SEQ_LEN    = C.MAX_SEQ_LEN

print(f"✅ [{TRACK}] vocab={VOCAB_SIZE:,}")
print(f"   Logits RAM: {8 * SEQ_LEN * VOCAB_SIZE * 4 / 1e6:.0f} MB")

CKPT_DIR = f"/content/drive/MyDrive/music_generator/checkpoints_v2/{TRACK}"
os.makedirs(CKPT_DIR, exist_ok=True)

MAX_STEPS    = C.MAX_STEPS
WARMUP_STEPS = C.WARMUP_STEPS
LR           = C.LEARNING_RATE
SAVE_EVERY   = C.SAVE_EVERY
LOG_EVERY    = C.LOG_EVERY
EVAL_EVERY   = C.EVAL_EVERY
PATIENCE     = C.PATIENCE
BATCH_SIZE   = C.BATCH_SIZE

def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * step / WARMUP_STEPS
    p = (step-WARMUP_STEPS)/(MAX_STEPS-WARMUP_STEPS)
    return LR*0.1 + 0.5*(LR-LR*0.1)*(1+math.cos(math.pi*p))

def save_ckpt(model, optimizer, step, loss, name):
    torch.save({"step":step,"model":model.state_dict(),
                "optimizer":optimizer.state_dict(),"loss":loss},
               os.path.join(CKPT_DIR, name))
    print(f"  💾 [{TRACK}] {name} step={step} loss={loss:.4f}")

@torch.no_grad()
def evaluate(model, loader, device, n=20):
    model.eval()
    total, count = 0.0, 0
    for x, y in loader:
        if count >= n: break
        _, loss = model(x.to(device), y.to(device))
        total += loss.item(); count += 1
    model.train()
    return total / max(count,1)

class TrackDataset(Dataset):
    def __init__(self, track):
        local = f"/content/music_generator/data/cache/{track}_chunks_v2.npy"
        drive = f"/content/drive/MyDrive/music_generator/cache/{track}_chunks_v2.npy"
        if not os.path.exists(local):
            import shutil; shutil.copy(drive, local)
            print(f"✅ Loaded {track} from Drive")
        self.data = np.load(local)
        # Clip any out-of-range tokens
        self.data = np.clip(self.data, 0, VOCAB_SIZE-1).astype(np.int32)
        print(f"✅ [{track}] {len(self.data):,} chunks | "
              f"{self.data.nbytes/1e6:.1f} MB")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        t = self.data[idx].astype(np.int64)
        x = torch.from_numpy(t[:-1])
        y = torch.from_numpy(t[1:].copy())
        y[y == PAD_ID] = -100
        return x, y

dataset  = TrackDataset(TRACK)
n_val    = max(1, int(len(dataset)*0.05))
train_ds, val_ds = random_split(dataset, [len(dataset)-n_val, n_val],
                                 generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, drop_last=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
print(f"✅ Train: {len(train_ds):,} | Val: {len(val_ds):,} | "
      f"Batch: {BATCH_SIZE} | Seq: {SEQ_LEN}")

device    = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.empty_cache()
model     = MusicTransformer(
    vocab_size=VOCAB_SIZE,
    max_len=SEQ_LEN
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                               weight_decay=C.WEIGHT_DECAY,
                               betas=(0.9,0.95))

print(f"\n✅ GPU: {torch.cuda.memory_allocated()/1e9:.2f}GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f}GB")

step, best_val, no_improve = 0, float("inf"), 0
train_iter   = iter(train_loader)
running_loss = 0.0
t0           = time.time()
EMOJI = {"bass":"🎸","chords":"🎹","melody":"🎵"}
print(f"\n{EMOJI[TRACK]} Training [{TRACK}] v2 | {device} | "
      f"vocab={VOCAB_SIZE:,} | 0→{MAX_STEPS}\n")

while step < MAX_STEPS:
    try:
        x, y = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        x, y = next(train_iter)

    x, y = x.to(device), y.to(device)
    lr = get_lr(step)
    for pg in optimizer.param_groups: pg["lr"] = lr

    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), C.GRAD_CLIP)
    optimizer.step()
    running_loss += loss.item()
    step += 1

    if step % LOG_EVERY == 0:
        avg     = running_loss / LOG_EVERY
        elapsed = time.time() - t0
        eta     = (MAX_STEPS-step)*elapsed/max(step,1)
        print(f"  step {step:>6}/{MAX_STEPS} | loss {avg:.4f} | "
              f"lr {lr:.2e} | {elapsed/60:.1f}m | ETA {eta/60:.0f}m")
        running_loss = 0.0

    if step % EVAL_EVERY == 0:
        val_loss = evaluate(model, val_loader, device)
        print(f"  📊 val {val_loss:.4f} | best {best_val:.4f}")
        if val_loss < best_val:
            best_val = val_loss; no_improve = 0
            save_ckpt(model, optimizer, step, val_loss, "best.pt")
        else:
            no_improve += 1
            print(f"  ⚠️  No improve {no_improve}/{PATIENCE}")
            if no_improve >= PATIENCE:
                print(f"🛑 Early stop! Best: {best_val:.4f}")
                break

    if step % SAVE_EVERY == 0:
        save_ckpt(model, optimizer, step, loss.item(),
                  f"step_{step:06d}.pt")

print(f"\n✅ [{TRACK}] Done! Best val: {best_val:.4f}")
print(f"   Saved → {CKPT_DIR}/best.pt")

✅ [bass] vocab=13,830
   Logits RAM: 113 MB
✅ [bass] 73,414 chunks | 75.2 MB
✅ Train: 69,744 | Val: 3,670 | Batch: 8 | Seq: 256
[MusicTransformer] 8.33M parameters

✅ GPU: 0.04GB / 15.64GB

🎸 Training [bass] v2 | cuda | vocab=13,830 | 0→50000

  step    100/50000 | loss 9.3964 | lr 4.95e-05 | 0.1m | ETA 56m
  step    200/50000 | loss 8.4776 | lr 9.95e-05 | 0.2m | ETA 47m
  step    300/50000 | loss 7.9354 | lr 1.50e-04 | 0.3m | ETA 43m
  step    400/50000 | loss 7.8335 | lr 1.99e-04 | 0.3m | ETA 42m
  step    500/50000 | loss 7.7408 | lr 2.49e-04 | 0.4m | ETA 41m
  📊 val 7.6246 | best inf
  💾 [bass] best.pt step=500 loss=7.6246
  step    600/50000 | loss 7.6493 | lr 2.99e-04 | 0.5m | ETA 41m
  step    700/50000 | loss 7.4398 | lr 3.50e-04 | 0.6m | ETA 41m
  step    800/50000 | loss 7.3229 | lr 4.00e-04 | 0.7m | ETA 40m
  step    900/50000 | loss 7.2576 | lr 4.50e-04 | 0.7m | ETA 40m
  step   1000/50000 | loss 7.1507 | lr 5.00e-04 | 0.8m | ETA 40m
  📊 val 7.0130 | best 7.6246
  💾 [bass] 

In [ ]:
import os, math, time, torch, torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
import sys
sys.path.insert(0, "/content/music_generator")
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["config","transformer"]):
        del sys.modules[mod]
import config as C
from model.transformer import MusicTransformer
import gc
torch.cuda.empty_cache(); gc.collect()

# ── CHANGE THIS ONLY ──
TRACK = "chords"
# ──────────────────────

VOCAB_SIZE = C.PITCHED_VOCAB_SIZE
PAD_ID     = C.PITCHED_SPECIALS["PAD"]
SEQ_LEN    = C.MAX_SEQ_LEN

print(f"✅ [{TRACK}] vocab={VOCAB_SIZE:,}")
print(f"   Logits RAM: {8 * SEQ_LEN * VOCAB_SIZE * 4 / 1e6:.0f} MB")

CKPT_DIR = f"/content/drive/MyDrive/music_generator/checkpoints_v2/{TRACK}"
os.makedirs(CKPT_DIR, exist_ok=True)

MAX_STEPS    = C.MAX_STEPS
WARMUP_STEPS = C.WARMUP_STEPS
LR           = C.LEARNING_RATE
SAVE_EVERY   = C.SAVE_EVERY
LOG_EVERY    = C.LOG_EVERY
EVAL_EVERY   = C.EVAL_EVERY
PATIENCE     = C.PATIENCE
BATCH_SIZE   = C.BATCH_SIZE

def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * step / WARMUP_STEPS
    p = (step-WARMUP_STEPS)/(MAX_STEPS-WARMUP_STEPS)
    return LR*0.1 + 0.5*(LR-LR*0.1)*(1+math.cos(math.pi*p))

def save_ckpt(model, optimizer, step, loss, name):
    torch.save({"step":step,"model":model.state_dict(),
                "optimizer":optimizer.state_dict(),"loss":loss},
               os.path.join(CKPT_DIR, name))
    print(f"  💾 [{TRACK}] {name} step={step} loss={loss:.4f}")

@torch.no_grad()
def evaluate(model, loader, device, n=20):
    model.eval()
    total, count = 0.0, 0
    for x, y in loader:
        if count >= n: break
        _, loss = model(x.to(device), y.to(device))
        total += loss.item(); count += 1
    model.train()
    return total / max(count,1)

class TrackDataset(Dataset):
    def __init__(self, track):
        local = f"/content/music_generator/data/cache/{track}_chunks_v2.npy"
        drive = f"/content/drive/MyDrive/music_generator/cache/{track}_chunks_v2.npy"
        if not os.path.exists(local):
            import shutil; shutil.copy(drive, local)
            print(f"✅ Loaded {track} from Drive")
        self.data = np.load(local)
        # Clip any out-of-range tokens
        self.data = np.clip(self.data, 0, VOCAB_SIZE-1).astype(np.int32)
        print(f"✅ [{track}] {len(self.data):,} chunks | "
              f"{self.data.nbytes/1e6:.1f} MB")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        t = self.data[idx].astype(np.int64)
        x = torch.from_numpy(t[:-1])
        y = torch.from_numpy(t[1:].copy())
        y[y == PAD_ID] = -100
        return x, y

dataset  = TrackDataset(TRACK)
n_val    = max(1, int(len(dataset)*0.05))
train_ds, val_ds = random_split(dataset, [len(dataset)-n_val, n_val],
                                 generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, drop_last=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
print(f"✅ Train: {len(train_ds):,} | Val: {len(val_ds):,} | "
      f"Batch: {BATCH_SIZE} | Seq: {SEQ_LEN}")

device    = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.empty_cache()
model     = MusicTransformer(
    vocab_size=VOCAB_SIZE,
    max_len=SEQ_LEN
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                               weight_decay=C.WEIGHT_DECAY,
                               betas=(0.9,0.95))

print(f"\n✅ GPU: {torch.cuda.memory_allocated()/1e9:.2f}GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f}GB")

step, best_val, no_improve = 0, float("inf"), 0
train_iter   = iter(train_loader)
running_loss = 0.0
t0           = time.time()
EMOJI = {"bass":"🎸","chords":"🎹","melody":"🎵"}
print(f"\n{EMOJI[TRACK]} Training [{TRACK}] v2 | {device} | "
      f"vocab={VOCAB_SIZE:,} | 0→{MAX_STEPS}\n")

while step < MAX_STEPS:
    try:
        x, y = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        x, y = next(train_iter)

    x, y = x.to(device), y.to(device)
    lr = get_lr(step)
    for pg in optimizer.param_groups: pg["lr"] = lr

    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), C.GRAD_CLIP)
    optimizer.step()
    running_loss += loss.item()
    step += 1

    if step % LOG_EVERY == 0:
        avg     = running_loss / LOG_EVERY
        elapsed = time.time() - t0
        eta     = (MAX_STEPS-step)*elapsed/max(step,1)
        print(f"  step {step:>6}/{MAX_STEPS} | loss {avg:.4f} | "
              f"lr {lr:.2e} | {elapsed/60:.1f}m | ETA {eta/60:.0f}m")
        running_loss = 0.0

    if step % EVAL_EVERY == 0:
        val_loss = evaluate(model, val_loader, device)
        print(f"  📊 val {val_loss:.4f} | best {best_val:.4f}")
        if val_loss < best_val:
            best_val = val_loss; no_improve = 0
            save_ckpt(model, optimizer, step, val_loss, "best.pt")
        else:
            no_improve += 1
            print(f"  ⚠️  No improve {no_improve}/{PATIENCE}")
            if no_improve >= PATIENCE:
                print(f"🛑 Early stop! Best: {best_val:.4f}")
                break

    if step % SAVE_EVERY == 0:
        save_ckpt(model, optimizer, step, loss.item(),
                  f"step_{step:06d}.pt")

print(f"\n✅ [{TRACK}] Done! Best val: {best_val:.4f}")
print(f"   Saved → {CKPT_DIR}/best.pt")

✅ [chords] vocab=13,830
   Logits RAM: 113 MB
✅ [chords] 8,021 chunks | 8.2 MB
✅ Train: 7,620 | Val: 401 | Batch: 8 | Seq: 256
[MusicTransformer] 8.33M parameters

✅ GPU: 0.23GB / 15.64GB

🎹 Training [chords] v2 | cuda | vocab=13,830 | 0→50000

  step    100/50000 | loss 9.4135 | lr 4.95e-05 | 0.1m | ETA 44m
  step    200/50000 | loss 8.5675 | lr 9.95e-05 | 0.2m | ETA 44m
  step    300/50000 | loss 7.9658 | lr 1.50e-04 | 0.3m | ETA 44m
  step    400/50000 | loss 7.7459 | lr 1.99e-04 | 0.4m | ETA 45m
  step    500/50000 | loss 7.6316 | lr 2.49e-04 | 0.5m | ETA 45m
  📊 val 7.5097 | best inf
  💾 [chords] best.pt step=500 loss=7.5097
  step    600/50000 | loss 7.3188 | lr 2.99e-04 | 0.6m | ETA 47m
  step    700/50000 | loss 7.0946 | lr 3.50e-04 | 0.7m | ETA 47m
  step    800/50000 | loss 6.9780 | lr 4.00e-04 | 0.8m | ETA 47m
  step    900/50000 | loss 6.8070 | lr 4.50e-04 | 0.9m | ETA 47m
  step   1000/50000 | loss 6.6975 | lr 5.00e-04 | 0.9m | ETA 47m
  📊 val 6.6614 | best 7.5097
  💾 [cho

In [ ]:
import os, math, time, torch, torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
import sys
sys.path.insert(0, "/content/music_generator")
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["config","transformer"]):
        del sys.modules[mod]
import config as C
from model.transformer import MusicTransformer
import gc
torch.cuda.empty_cache(); gc.collect()

# ── CHANGE THIS ONLY ──
TRACK = "melody"
# ──────────────────────

VOCAB_SIZE = C.PITCHED_VOCAB_SIZE
PAD_ID     = C.PITCHED_SPECIALS["PAD"]
SEQ_LEN    = C.MAX_SEQ_LEN

print(f"✅ [{TRACK}] vocab={VOCAB_SIZE:,}")
print(f"   Logits RAM: {8 * SEQ_LEN * VOCAB_SIZE * 4 / 1e6:.0f} MB")

CKPT_DIR = f"/content/drive/MyDrive/music_generator/checkpoints_v2/{TRACK}"
os.makedirs(CKPT_DIR, exist_ok=True)

MAX_STEPS    = C.MAX_STEPS
WARMUP_STEPS = C.WARMUP_STEPS
LR           = C.LEARNING_RATE
SAVE_EVERY   = C.SAVE_EVERY
LOG_EVERY    = C.LOG_EVERY
EVAL_EVERY   = C.EVAL_EVERY
PATIENCE     = C.PATIENCE
BATCH_SIZE   = C.BATCH_SIZE

def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * step / WARMUP_STEPS
    p = (step-WARMUP_STEPS)/(MAX_STEPS-WARMUP_STEPS)
    return LR*0.1 + 0.5*(LR-LR*0.1)*(1+math.cos(math.pi*p))

def save_ckpt(model, optimizer, step, loss, name):
    torch.save({"step":step,"model":model.state_dict(),
                "optimizer":optimizer.state_dict(),"loss":loss},
               os.path.join(CKPT_DIR, name))
    print(f"  💾 [{TRACK}] {name} step={step} loss={loss:.4f}")

@torch.no_grad()
def evaluate(model, loader, device, n=20):
    model.eval()
    total, count = 0.0, 0
    for x, y in loader:
        if count >= n: break
        _, loss = model(x.to(device), y.to(device))
        total += loss.item(); count += 1
    model.train()
    return total / max(count,1)

class TrackDataset(Dataset):
    def __init__(self, track):
        local = f"/content/music_generator/data/cache/{track}_chunks_v2.npy"
        drive = f"/content/drive/MyDrive/music_generator/cache/{track}_chunks_v2.npy"
        if not os.path.exists(local):
            import shutil; shutil.copy(drive, local)
            print(f"✅ Loaded {track} from Drive")
        self.data = np.load(local)
        # Clip any out-of-range tokens
        self.data = np.clip(self.data, 0, VOCAB_SIZE-1).astype(np.int32)
        print(f"✅ [{track}] {len(self.data):,} chunks | "
              f"{self.data.nbytes/1e6:.1f} MB")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        t = self.data[idx].astype(np.int64)
        x = torch.from_numpy(t[:-1])
        y = torch.from_numpy(t[1:].copy())
        y[y == PAD_ID] = -100
        return x, y

dataset  = TrackDataset(TRACK)
n_val    = max(1, int(len(dataset)*0.05))
train_ds, val_ds = random_split(dataset, [len(dataset)-n_val, n_val],
                                 generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, drop_last=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
print(f"✅ Train: {len(train_ds):,} | Val: {len(val_ds):,} | "
      f"Batch: {BATCH_SIZE} | Seq: {SEQ_LEN}")

device    = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.empty_cache()
model     = MusicTransformer(
    vocab_size=VOCAB_SIZE,
    max_len=SEQ_LEN
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                               weight_decay=C.WEIGHT_DECAY,
                               betas=(0.9,0.95))

print(f"\n✅ GPU: {torch.cuda.memory_allocated()/1e9:.2f}GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f}GB")

step, best_val, no_improve = 0, float("inf"), 0
train_iter   = iter(train_loader)
running_loss = 0.0
t0           = time.time()
EMOJI = {"bass":"🎸","chords":"🎹","melody":"🎵"}
print(f"\n{EMOJI[TRACK]} Training [{TRACK}] v2 | {device} | "
      f"vocab={VOCAB_SIZE:,} | 0→{MAX_STEPS}\n")

while step < MAX_STEPS:
    try:
        x, y = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        x, y = next(train_iter)

    x, y = x.to(device), y.to(device)
    lr = get_lr(step)
    for pg in optimizer.param_groups: pg["lr"] = lr

    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), C.GRAD_CLIP)
    optimizer.step()
    running_loss += loss.item()
    step += 1

    if step % LOG_EVERY == 0:
        avg     = running_loss / LOG_EVERY
        elapsed = time.time() - t0
        eta     = (MAX_STEPS-step)*elapsed/max(step,1)
        print(f"  step {step:>6}/{MAX_STEPS} | loss {avg:.4f} | "
              f"lr {lr:.2e} | {elapsed/60:.1f}m | ETA {eta/60:.0f}m")
        running_loss = 0.0

    if step % EVAL_EVERY == 0:
        val_loss = evaluate(model, val_loader, device)
        print(f"  📊 val {val_loss:.4f} | best {best_val:.4f}")
        if val_loss < best_val:
            best_val = val_loss; no_improve = 0
            save_ckpt(model, optimizer, step, val_loss, "best.pt")
        else:
            no_improve += 1
            print(f"  ⚠️  No improve {no_improve}/{PATIENCE}")
            if no_improve >= PATIENCE:
                print(f"🛑 Early stop! Best: {best_val:.4f}")
                break

    if step % SAVE_EVERY == 0:
        save_ckpt(model, optimizer, step, loss.item(),
                  f"step_{step:06d}.pt")

print(f"\n✅ [{TRACK}] Done! Best val: {best_val:.4f}")
print(f"   Saved → {CKPT_DIR}/best.pt")

✅ [melody] vocab=13,830
   Logits RAM: 113 MB
✅ [melody] 10,118 chunks | 10.4 MB
✅ Train: 9,613 | Val: 505 | Batch: 8 | Seq: 256
[MusicTransformer] 8.33M parameters

✅ GPU: 0.04GB / 15.64GB

🎵 Training [melody] v2 | cuda | vocab=13,830 | 0→50000

  step    100/50000 | loss 9.4013 | lr 4.95e-05 | 0.1m | ETA 47m
  step    200/50000 | loss 8.4945 | lr 9.95e-05 | 0.2m | ETA 42m
  step    300/50000 | loss 7.9039 | lr 1.50e-04 | 0.2m | ETA 40m
  step    400/50000 | loss 7.7093 | lr 1.99e-04 | 0.3m | ETA 39m
  step    500/50000 | loss 7.2555 | lr 2.49e-04 | 0.4m | ETA 39m
  📊 val 6.9400 | best inf
  💾 [melody] best.pt step=500 loss=6.9400
  step    600/50000 | loss 6.7842 | lr 2.99e-04 | 0.5m | ETA 39m
  step    700/50000 | loss 6.4672 | lr 3.50e-04 | 0.6m | ETA 39m
  step    800/50000 | loss 6.3367 | lr 4.00e-04 | 0.6m | ETA 39m
  step    900/50000 | loss 6.2657 | lr 4.50e-04 | 0.7m | ETA 39m
  step   1000/50000 | loss 6.1846 | lr 5.00e-04 | 0.8m | ETA 38m
  📊 val 6.1959 | best 6.9400
  💾 [m

In [ ]:
import sys, os, torch

sys.path.insert(0, "/content/music_generator")

# Reload all modules
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["combiner","synchronizer","generate","text_encoder","transformer","config"]):
        del sys.modules[mod]

import config as C
from model.transformer import MusicTransformer
from generate import MusicGenerationPipeline

# Load pipeline
pipeline = MusicGenerationPipeline()
CKPT_BASE = "/content/drive/MyDrive/music_generator/checkpoints_v2"

# Load drums with auto-detect
drum_ckpt = torch.load(f"{CKPT_BASE}/drums/best.pt", map_location=pipeline.device)
sd = drum_ckpt["model"]
drum_model = MusicTransformer(
    vocab_size = sd["token_emb.weight"].shape[0],
    max_len    = sd["pos_emb.weight"].shape[0],
    d_model    = sd["token_emb.weight"].shape[1],
    n_layers   = sum(1 for k in sd if k.startswith("blocks.") and k.endswith(".ln1.weight")),
    n_heads=C.N_HEADS, d_ff=C.D_FF, dropout=0.0
).to(pipeline.device)
drum_model.load_state_dict(sd)
drum_model.eval()
pipeline.models["drums"] = drum_model
print("✅ [drums] loaded")

for track in ["bass","chords","melody"]:
    pipeline.load_model(track, f"{CKPT_BASE}/{track}/best.pt")

print(f"\n✅ All 4 models loaded: {list(pipeline.models.keys())}")

ModuleNotFoundError: No module named 'generate'

In [ ]:
import os, sys, torch

# Create folders
for d in ["/content/music_generator","/content/music_generator/data",
          "/content/music_generator/model","/content/music_generator/midi_utils"]:
    os.makedirs(d, exist_ok=True)
for d in ["/content/music_generator/data","/content/music_generator/model",
          "/content/music_generator/midi_utils"]:
    open(f"{d}/__init__.py","w").close()

sys.path.insert(0, "/content/music_generator")

# config.py
open("/content/music_generator/config.py","w").write("""
import os
BASE_DIR="/content/music_generator"
DATA_DIR=os.path.join(BASE_DIR,"data")
CACHE_DIR=os.path.join(DATA_DIR,"cache")
DRIVE_CACHE="/content/drive/MyDrive/music_generator/cache"
DRIVE_CKPT="/content/drive/MyDrive/music_generator/checkpoints_v2"
N_VELOCITY_BINS=4;N_TIME_BINS=12;N_DURATION_BINS=4;N_PITCH_BINS=72
MIDI_PITCH_MIN=24;MIDI_PITCH_MAX=96
DRUM_PITCH_MAP={36:0,38:1,40:1,42:2,44:2,46:3,49:4,51:5,53:5,47:6,48:6,45:7,43:7,41:8}
N_DRUM_INSTRUMENTS=9
DRUM_CONTENT_VOCAB=N_DRUM_INSTRUMENTS*N_VELOCITY_BINS*N_TIME_BINS
PITCHED_CONTENT_VOCAB=N_PITCH_BINS*N_VELOCITY_BINS*N_TIME_BINS*N_DURATION_BINS
N_SPECIALS=6
DRUM_VOCAB_SIZE=DRUM_CONTENT_VOCAB+N_SPECIALS
PITCHED_VOCAB_SIZE=PITCHED_CONTENT_VOCAB+N_SPECIALS
def make_specials(n):
    return{"BOS":n+0,"EOS":n+1,"PAD":n+2,"UNCOND":n+3,"COND":n+4,"MARKER":n+5}
DRUM_SPECIALS=make_specials(DRUM_CONTENT_VOCAB)
PITCHED_SPECIALS=make_specials(PITCHED_CONTENT_VOCAB)
BOS_TOKEN=DRUM_SPECIALS["BOS"];EOS_TOKEN=DRUM_SPECIALS["EOS"]
PAD_TOKEN=DRUM_SPECIALS["PAD"];UNCOND_TOKEN=DRUM_SPECIALS["UNCOND"]
COND_TOKEN=DRUM_SPECIALS["COND"];VOCAB_SIZE=DRUM_VOCAB_SIZE
D_MODEL=256;N_HEADS=4;N_LAYERS=6;D_FF=1024;DROPOUT=0.2
MAX_SEQ_LEN=256;CONTEXT_LEN=256;BATCH_SIZE=8
LEARNING_RATE=5e-4;WEIGHT_DECAY=0.01;WARMUP_STEPS=500
MAX_STEPS=15000;GRAD_CLIP=1.0;SAVE_EVERY=1000
LOG_EVERY=50;EVAL_EVERY=200;PATIENCE=10
GEN_TEMPERATURE=1.0;GEN_TOP_P=0.92;GEN_MAX_TOKENS=512
BASS_PROGRAM=32;CHORD_PROGRAM=0;MELODY_PROGRAM=0
TEXT_EMB_DIM=384;TEXT_MODEL="all-MiniLM-L6-v2"
""")

# transformer.py
open("/content/music_generator/model/transformer.py","w").write('''
import math,torch,torch.nn as nn,torch.nn.functional as F,sys
sys.path.insert(0,"/content/music_generator")
import config as C
class CausalSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout,max_len):
        super().__init__()
        self.n_heads=n_heads;self.d_head=d_model//n_heads;self.scale=self.d_head**-0.5
        self.qkv_proj=nn.Linear(d_model,3*d_model,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_drop=nn.Dropout(dropout);self.resid_drop=nn.Dropout(dropout)
        self.register_buffer("mask",torch.tril(torch.ones(max_len,max_len)).unsqueeze(0).unsqueeze(0))
    def forward(self,x):
        B,T,C=x.shape
        q,k,v=self.qkv_proj(x).chunk(3,dim=-1)
        q=q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v=v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        attn=(q@k.transpose(-2,-1))*self.scale
        attn=attn.masked_fill(self.mask[:,:,:T,:T]==0,float("-inf"))
        attn=self.attn_drop(F.softmax(attn,dim=-1))
        return self.resid_drop(self.out_proj((attn@v).transpose(1,2).contiguous().view(B,T,C)))
class FeedForward(nn.Module):
    def __init__(self,d_model,d_ff,dropout):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d_model,d_ff,bias=False),nn.GELU(),
            nn.Dropout(dropout),nn.Linear(d_ff,d_model,bias=False),nn.Dropout(dropout))
    def forward(self,x): return self.net(x)
class TransformerBlock(nn.Module):
    def __init__(self,d_model,n_heads,d_ff,dropout,max_len):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model);self.attn=CausalSelfAttention(d_model,n_heads,dropout,max_len)
        self.ln2=nn.LayerNorm(d_model);self.ff=FeedForward(d_model,d_ff,dropout)
    def forward(self,x):
        x=x+self.attn(self.ln1(x));x=x+self.ff(self.ln2(x));return x
class MusicTransformer(nn.Module):
    def __init__(self,vocab_size=None,d_model=None,n_heads=None,n_layers=None,d_ff=None,dropout=None,max_len=None):
        super().__init__()
        vocab_size=vocab_size or C.PITCHED_VOCAB_SIZE;d_model=d_model or C.D_MODEL
        n_heads=n_heads or C.N_HEADS;n_layers=n_layers or C.N_LAYERS
        d_ff=d_ff or C.D_FF;dropout=dropout or C.DROPOUT;max_len=max_len or C.MAX_SEQ_LEN
        self.vocab_size=vocab_size
        self.token_emb=nn.Embedding(vocab_size,d_model);self.pos_emb=nn.Embedding(max_len,d_model)
        self.drop=nn.Dropout(dropout)
        self.blocks=nn.ModuleList([TransformerBlock(d_model,n_heads,d_ff,dropout,max_len) for _ in range(n_layers)])
        self.ln_f=nn.LayerNorm(d_model);self.head=nn.Linear(d_model,vocab_size,bias=False)
        self.head.weight=self.token_emb.weight;self._init_weights()
        print(f"[MusicTransformer] {sum(p.numel() for p in self.parameters())/1e6:.2f}M params | vocab={vocab_size:,}")
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.normal_(m.weight,0,0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0,0.02)
            elif isinstance(m,nn.LayerNorm): nn.init.ones_(m.weight);nn.init.zeros_(m.bias)
    def forward(self,input_ids,labels=None):
        B,T=input_ids.shape;pos=torch.arange(T,device=input_ids.device).unsqueeze(0)
        x=self.drop(self.token_emb(input_ids)+self.pos_emb(pos))
        for block in self.blocks: x=block(x)
        logits=self.head(self.ln_f(x));loss=None
        if labels is not None:
            loss=F.cross_entropy(logits.view(-1,self.vocab_size),labels.view(-1),ignore_index=-100)
        return logits,loss
''')

# synchronizer.py
open("/content/music_generator/midi_utils/synchronizer.py","w").write("""
import pretty_midi
def quantize_to_grid(notes,tempo,subdivisions=8):
    bd=60.0/max(tempo,1.0);gd=bd/subdivisions;out=[]
    for n in notes:
        gp=round(n.start/gd);start=gp*gd;dg=max(2,round((n.end-n.start)/gd))
        out.append(pretty_midi.Note(velocity=n.velocity,pitch=n.pitch,start=start,end=start+dg*gd))
    return out
def remove_overlaps(notes):
    notes.sort(key=lambda n:(n.pitch,n.start));fixed=[];last_end={}
    for note in notes:
        p=note.pitch;start=max(note.start,last_end.get(p,0));end=max(start+0.05,note.end)
        fixed.append(pretty_midi.Note(velocity=note.velocity,pitch=p,start=start,end=end));last_end[p]=end
    return fixed
def thin_notes(notes,max_per_second=4):
    if not notes: return notes
    notes.sort(key=lambda n:n.start);kept=[notes[0]]
    for note in notes[1:]:
        if note.start-kept[-1].start>=1.0/max_per_second: kept.append(note)
    return kept
def loop_to_bars(notes,tempo,bars=8,bpb=4):
    if not notes: return notes
    bd=60.0/max(tempo,1.0);tt=bars*bpb*bd;te=max(n.end for n in notes)
    if te>=tt: return [n for n in notes if n.start<tt]
    looped=list(notes);offset=te
    while offset<tt:
        for n in notes:
            ns=n.start+offset
            if ns>=tt: break
            looped.append(pretty_midi.Note(velocity=n.velocity,pitch=n.pitch,start=ns,end=min(n.end+offset,tt)))
        offset+=te
    return looped
def synchronize_tracks(tracks_dict,tempo,bars=8):
    synced={}
    for name,notes in tracks_dict.items():
        if not notes: synced[name]=[]; continue
        notes=quantize_to_grid(notes,tempo);notes=remove_overlaps(notes)
        notes=thin_notes(notes,max_per_second=4);notes=loop_to_bars(notes,tempo,bars)
        notes.sort(key=lambda n:n.start);synced[name]=notes
        dur=max(n.end for n in notes) if notes else 0
        print(f"  ✅ {name:8}: {len(notes):>4} notes | {dur:.1f}s")
    return synced
""")

# combiner.py
open("/content/music_generator/midi_utils/combiner.py","w").write("""
import pretty_midi,sys
sys.path.insert(0,"/content/music_generator")
import config as C
from midi_utils.synchronizer import synchronize_tracks
class MIDICombiner:
    GM_DRUMS=[36,38,42,46,49,51,47,45,41]
    def __init__(self):
        self.nt=C.N_TIME_BINS;self.nv=C.N_VELOCITY_BINS;self.nd=C.N_DURATION_BINS
    def decode_pitched(self,t):
        if t>=C.PITCHED_CONTENT_VOCAB or t<0: return None
        db=t%self.nd;t//=self.nd;tb=t%self.nt;t//=self.nt;vb=t%self.nv
        pitch=(t//self.nv)+C.MIDI_PITCH_MIN
        if not(C.MIDI_PITCH_MIN<=pitch<=C.MIDI_PITCH_MAX): return None
        return pitch,vb,tb,db
    def to_drum_notes(self,tokens,tempo):
        bd=60.0/max(tempo,1.0);sd=bd/self.nt
        sp=set(C.PITCHED_SPECIALS.values());notes=[];beat_groups=[[]];prev_tb=-1
        for tok in tokens:
            if tok in sp: continue
            d=self.decode_pitched(tok)
            if d is None: continue
            pitch,vb,tb,db=d
            if prev_tb>self.nt*0.5 and tb<self.nt*0.3: beat_groups.append([])
            beat_groups[-1].append((pitch,vb,tb));prev_tb=tb
        for beat_idx,group in enumerate(beat_groups):
            for pitch,vb,tb in group:
                start=beat_idx*bd+tb*sd
                velocity=max(1,min(127,int((vb+0.5)/self.nv*127)))
                gm_pitch=min(self.GM_DRUMS,key=lambda x:abs(x-pitch))
                notes.append(pretty_midi.Note(velocity=velocity,pitch=gm_pitch,start=start,end=start+0.05))
        return notes
    def to_pitched_notes(self,tokens,tempo):
        bd=60.0/max(tempo,1.0);sd=bd/self.nt
        sp=set(C.PITCHED_SPECIALS.values());notes=[];beat_groups=[[]];prev_tb=-1
        for tok in tokens:
            if tok in sp: continue
            d=self.decode_pitched(tok)
            if d is None: continue
            pitch,vb,tb,db=d
            if prev_tb>self.nt*0.5 and tb<self.nt*0.3: beat_groups.append([])
            beat_groups[-1].append((pitch,vb,tb,db));prev_tb=tb
        for beat_idx,group in enumerate(beat_groups):
            for pitch,vb,tb,db in group:
                start=beat_idx*bd+tb*sd
                duration=max(0.15,(db+1)*sd*(self.nt/self.nd)*2.0)
                velocity=max(1,min(127,int((vb+0.5)/self.nv*127)))
                notes.append(pretty_midi.Note(velocity=velocity,pitch=pitch,start=start,end=start+duration))
        return notes
    def apply_mood_velocity(self,notes,mood):
        if not notes: return notes
        vel_ranges={"calm":(40,65),"happy":(65,95),"sad":(35,60),"intense":(85,115),"romantic":(45,70),"epic":(85,115),"neutral":(55,80)}
        lo,hi=vel_ranges.get(mood,(55,80))
        return [pretty_midi.Note(velocity=max(1,min(127,int(lo+(n.velocity/127)*(hi-lo)))),pitch=n.pitch,start=n.start,end=n.end) for n in notes]
    def combine(self,drum_tokens,bass_tokens,chord_tokens,melody_tokens,tempo=120.0,bars=8,mood="neutral",output_path="/content/output.mid"):
        pm=pretty_midi.PrettyMIDI(initial_tempo=tempo)
        raw={"drums":self.to_drum_notes(drum_tokens,tempo),"bass":self.to_pitched_notes(bass_tokens,tempo),
             "chords":self.to_pitched_notes(chord_tokens,tempo),"melody":self.to_pitched_notes(melody_tokens,tempo)}
        for name in raw:
            raw[name]=self.apply_mood_velocity(raw[name],mood)
            print(f"  {name}: {len(raw[name])} notes")
        synced=synchronize_tracks(raw,tempo,bars)
        for name,prog,is_drum,label in [("drums",0,True,"Drums"),("bass",C.BASS_PROGRAM,False,"Bass"),
                                         ("chords",C.CHORD_PROGRAM,False,"Chords"),("melody",C.MELODY_PROGRAM,False,"Melody")]:
            inst=pretty_midi.Instrument(program=prog,is_drum=is_drum,name=label)
            inst.notes=synced.get(name,[]);pm.instruments.append(inst)
        pm.write(output_path)
        total=sum(len(i.notes) for i in pm.instruments)
        print(f"✅ {output_path} | {tempo}BPM | {pm.get_end_time():.1f}s | {total} notes")
        return output_path
""")

# text_encoder.py
open("/content/music_generator/data/text_encoder.py","w").write("""
import torch,torch.nn as nn,sys
sys.path.insert(0,"/content/music_generator")
import config as C
class TextEncoder(nn.Module):
    def __init__(self,device="cpu"):
        super().__init__()
        from sentence_transformers import SentenceTransformer
        self.encoder=SentenceTransformer(C.TEXT_MODEL)
        self.projector=nn.Linear(C.TEXT_EMB_DIM,C.D_MODEL)
        self.device=device
    def get_musical_params(self,prompt):
        p=prompt.lower()
        if any(w in p for w in ["study","focus","work","reading"]): tempo=75
        elif any(w in p for w in ["sleep","lullaby","bedtime","dreamy"]): tempo=60
        elif any(w in p for w in ["meditation","meditate","zen","yoga"]): tempo=65
        elif any(w in p for w in ["workout","gym","running","pump up"]): tempo=155
        elif any(w in p for w in ["party","dance","club","disco"]): tempo=128
        elif any(w in p for w in ["sad","heartbreak","lonely","melancholy"]): tempo=65
        elif any(w in p for w in ["happy","joy","celebration","cheerful"]): tempo=130
        elif any(w in p for w in ["romantic","love","intimate"]): tempo=80
        elif any(w in p for w in ["angry","intense","aggressive","metal"]): tempo=160
        elif any(w in p for w in ["chill","relax","lofi","lo-fi","cozy"]): tempo=80
        elif any(w in p for w in ["epic","cinematic","dramatic","grand"]): tempo=100
        else: tempo=110
        if any(w in p for w in ["fast","upbeat","quick"]): tempo=min(tempo+20,180)
        elif any(w in p for w in ["slow","gentle","soft"]): tempo=max(tempo-20,55)
        if "jazz" in p: tempo=int(tempo*1.05)
        if "hip hop" in p: tempo=90
        if "lofi" in p or "lo-fi" in p: tempo=80
        if "ballad" in p: tempo=max(int(tempo*.7),55)
        if "metal" in p: tempo=170
        tempo=max(50,min(200,tempo))
        if any(w in p for w in ["happy","joy","workout","gym","celebration"]): mood="happy"
        elif any(w in p for w in ["sad","melancholy","cry","heartbreak","dark","ballad"]): mood="sad"
        elif any(w in p for w in ["intense","aggressive","metal","rage"]): mood="intense"
        elif any(w in p for w in ["calm","chill","soft","meditation","study","sleep","lofi"]): mood="calm"
        elif any(w in p for w in ["romantic","love","intimate"]): mood="romantic"
        elif any(w in p for w in ["epic","cinematic","dramatic","grand"]): mood="epic"
        else: mood="neutral"
        bars=4 if any(w in p for w in ["short","quick","brief"]) else 16 if any(w in p for w in ["long","extended","full"]) else 8
        return {"tempo":tempo,"mood":mood,"bars":bars}
""")

# generate.py
open("/content/music_generator/generate.py","w").write("""
import torch,torch.nn.functional as F,sys,os
sys.path.insert(0,"/content/music_generator")
import config as C
from model.transformer import MusicTransformer
from midi_utils.combiner import MIDICombiner
from data.text_encoder import TextEncoder
TRACK_PARAMS={"drums":{"temperature":0.85,"top_p":0.90,"top_k":40},"bass":{"temperature":0.80,"top_p":0.85,"top_k":30},"chords":{"temperature":0.90,"top_p":0.90,"top_k":50},"melody":{"temperature":0.75,"top_p":0.80,"top_k":20}}
ALL_TRACKS=["drums","bass","chords","melody"]
class MusicGenerationPipeline:
    def __init__(self,device=None):
        self.device=device or("cuda" if torch.cuda.is_available() else "cpu")
        self.combiner=MIDICombiner();self.encoder=TextEncoder(device=self.device);self.models={}
        print(f"[Pipeline] Device: {self.device}")
    def load_model(self,track,ckpt_path):
        model=MusicTransformer(vocab_size=C.PITCHED_VOCAB_SIZE,max_len=C.MAX_SEQ_LEN).to(self.device)
        ckpt=torch.load(ckpt_path,map_location=self.device)
        model.load_state_dict(ckpt["model"]);model.eval();self.models[track]=model
        print(f"  ✅ [{track}] step={ckpt['step']} | loss={ckpt['loss']:.4f}")
    def load_all_models(self,ckpt_base):
        for track in ALL_TRACKS:
            path=os.path.join(ckpt_base,track,"best.pt")
            if os.path.exists(path): self.load_model(track,path)
    @torch.no_grad()
    def sample_token(self,logits,temperature,top_p,top_k):
        logits=logits/max(temperature,1e-8)
        if top_k>0:
            kth_val=torch.topk(logits,min(top_k,logits.size(-1)))[0][...,-1,None]
            logits=logits.masked_fill(logits<kth_val,float("-inf"))
        probs=F.softmax(logits,dim=-1);sp2,si=torch.sort(probs,descending=True)
        cs=torch.cumsum(sp2,dim=-1);sp2[cs-sp2>top_p]=0.0;sp2/=sp2.sum()
        return si[torch.multinomial(sp2,1)].item()
    @torch.no_grad()
    def generate_track(self,track,max_tokens=256,creativity=1.0):
        if track not in self.models: return []
        d=TRACK_PARAMS[track]
        temp=max(0.5,min(2.0,d["temperature"]*creativity))
        tp=d["top_p"];tk=max(5,int(d["top_k"]*creativity))
        model=self.models[track];sp=C.PITCHED_SPECIALS;vocab=C.PITCHED_VOCAB_SIZE
        ctx=[sp["BOS"],sp["UNCOND"],sp["MARKER"]]
        for _ in range(max_tokens):
            ids=torch.tensor([ctx[-C.CONTEXT_LEN:]],dtype=torch.long,device=self.device)
            logits,_=model(ids);nl=logits[0,-1,:].clone()
            for v in sp.values():
                if v!=sp["EOS"] and v<vocab: nl[v]=float("-inf")
            nt=self.sample_token(nl,temp,tp,tk);ctx.append(nt)
            if nt==sp["EOS"]: break
        return ctx
    def generate(self,prompt="happy jazz",output_path="/content/out.mid",
                 max_tokens=256,creativity=1.0,tempo=None,bars=None,tracks=None):
        print(f"\\n🎵 '{prompt}'")
        active=tracks or ALL_TRACKS
        params=self.encoder.get_musical_params(prompt)
        tempo=tempo or params["tempo"];bars=bars or params["bars"];mood=params["mood"]
        print(f"   Tempo={tempo} | Bars={bars} | Mood={mood} | Tracks={active}")
        all_tokens={t:[] for t in ALL_TRACKS}
        for track in active:
            print(f"   {track}...",end=" ",flush=True)
            all_tokens[track]=self.generate_track(track,max_tokens,creativity)
            print(f"{len(all_tokens[track])} tokens ✅")
        return self.combiner.combine(
            drum_tokens=all_tokens["drums"],bass_tokens=all_tokens["bass"],
            chord_tokens=all_tokens["chords"],melody_tokens=all_tokens["melody"],
            tempo=tempo,bars=bars,mood=mood,output_path=output_path)
""")

print("✅ All files written!")

# Reload modules
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["combiner","synchronizer","generate","text_encoder","transformer","config"]):
        del sys.modules[mod]

import config as C
from model.transformer import MusicTransformer
from generate import MusicGenerationPipeline

# Load pipeline
pipeline = MusicGenerationPipeline()
CKPT_BASE = "/content/drive/MyDrive/music_generator/checkpoints_v2"

# Load drums
drum_ckpt = torch.load(f"{CKPT_BASE}/drums/best.pt", map_location=pipeline.device)
sd = drum_ckpt["model"]
drum_model = MusicTransformer(
    vocab_size=sd["token_emb.weight"].shape[0],
    max_len=sd["pos_emb.weight"].shape[0],
    d_model=sd["token_emb.weight"].shape[1],
    n_layers=sum(1 for k in sd if k.startswith("blocks.") and k.endswith(".ln1.weight")),
    n_heads=C.N_HEADS, d_ff=C.D_FF, dropout=0.0
).to(pipeline.device)
drum_model.load_state_dict(sd)
drum_model.eval()
pipeline.models["drums"] = drum_model
print("✅ [drums] loaded")

for track in ["bass","chords","melody"]:
    pipeline.load_model(track, f"{CKPT_BASE}/{track}/best.pt")

print(f"\n✅ All 4 models loaded: {list(pipeline.models.keys())}")

✅ All files written!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Pipeline] Device: cuda
[MusicTransformer] 8.33M params | vocab=13,830
✅ [drums] loaded
[MusicTransformer] 8.33M params | vocab=13,830
  ✅ [bass] step=49000 | loss=2.3091
[MusicTransformer] 8.33M params | vocab=13,830
  ✅ [chords] step=50000 | loss=1.3857
[MusicTransformer] 8.33M params | vocab=13,830
  ✅ [melody] step=50000 | loss=1.5478

✅ All 4 models loaded: ['drums', 'bass', 'chords', 'melody']


In [ ]:
import sys
sys.path.insert(0, "/content/music_generator")

test_prompts = [
    ("lofi hip hop to chill",      "/content/lofi.mid"),
    ("epic cinematic soundtrack",  "/content/epic.mid"),
    ("sad music for a rainy day",  "/content/sad.mid"),
    ("pump up workout music",      "/content/workout.mid"),
    ("soft jazz for coffee shop",  "/content/jazz.mid"),
]

print("Generating test samples...")
for prompt, path in test_prompts:
    pipeline.generate(prompt=prompt, output_path=path, bars=8)
    print(f"✅ {prompt}")

print("\n✅ All 5 samples generated!")

Generating test samples...

🎵 'lofi hip hop to chill'
   Tempo=80 | Bars=8 | Mood=calm | Tracks=['drums', 'bass', 'chords', 'melody']
   drums... 19 tokens ✅
   bass... 259 tokens ✅
   chords... 259 tokens ✅
   melody... 259 tokens ✅
  drums: 15 notes
  bass: 256 notes
  chords: 256 notes
  melody: 256 notes
  ✅ drums   :   82 notes | 24.0s
  ✅ bass    :   51 notes | 24.5s
  ✅ chords  :   69 notes | 23.7s
  ✅ melody  :   61 notes | 24.1s
✅ /content/lofi.mid | 80BPM | 24.5s | 263 notes
✅ lofi hip hop to chill

🎵 'epic cinematic soundtrack'
   Tempo=100 | Bars=8 | Mood=epic | Tracks=['drums', 'bass', 'chords', 'melody']
   drums... 259 tokens ✅
   bass... 259 tokens ✅
   chords... 259 tokens ✅
   melody... 259 tokens ✅
  drums: 256 notes
  bass: 256 notes
  chords: 256 notes
  melody: 256 notes
  ✅ drums   :   51 notes | 19.0s
  ✅ bass    :   61 notes | 19.2s
  ✅ chords  :   54 notes | 19.1s
  ✅ melody  :   47 notes | 20.2s
✅ /content/epic.mid | 100BPM | 20.2s | 213 notes
✅ epic cinemati

In [ ]:
import pretty_midi, numpy as np, math

def evaluate_midi(path, prompt=""):
    pm = pretty_midi.PrettyMIDI(path)
    _, tempos = pm.get_tempo_changes()
    tempo = tempos[0] if len(tempos) > 0 else 120.0
    duration = pm.get_end_time()

    print(f"\n{'='*60}")
    print(f"Prompt:   '{prompt}'")
    print(f"Tempo:    {tempo:.0f} BPM | Duration: {duration:.1f}s")
    print(f"{'='*60}")

    results = {}
    for inst in pm.instruments:
        notes = inst.notes
        if not notes: continue
        pitches    = [n.pitch for n in notes]
        durations  = [n.end - n.start for n in notes]
        velocities = [n.velocity for n in notes]
        starts     = sorted([n.start for n in notes])
        density    = len(notes) / max(duration, 1)
        pitch_range= max(pitches) - min(pitches)
        intervals  = [abs(pitches[i+1]-pitches[i]) for i in range(len(pitches)-1)]
        avg_interval = np.mean(intervals) if intervals else 0
        pc_hist = np.zeros(12)
        for p in pitches: pc_hist[p%12] += 1
        pc_hist = pc_hist/pc_hist.sum()
        pc_hist = pc_hist[pc_hist>0]
        entropy = -np.sum(pc_hist * np.log2(pc_hist))
        gaps = [starts[i+1]-starts[i] for i in range(len(starts)-1)]
        overlaps = sum(1 for g in gaps if g < 0)

        print(f"\n  [{inst.name}]")
        print(f"  Notes:         {len(notes)}")
        print(f"  Density:       {density:.2f} notes/sec")
        print(f"  Pitch range:   {pitch_range} semitones")
        print(f"  Avg interval:  {avg_interval:.1f} semi  {'✅ smooth' if avg_interval<4 else '⚠️ jumpy' if avg_interval>7 else '✅ ok'}")
        print(f"  Velocity avg:  {np.mean(velocities):.0f}")
        print(f"  PC entropy:    {entropy:.2f} bits  {'✅' if 2.5<=entropy<=3.5 else ''}")
        print(f"  Overlaps:      {overlaps}  {'✅' if overlaps==0 else '❌'}")

        results[inst.name] = {
            "notes": len(notes), "density": round(density,2),
            "pitch_range": pitch_range, "avg_interval": round(avg_interval,2),
            "velocity_avg": round(np.mean(velocities)),
            "pc_entropy": round(entropy,2), "overlaps": overlaps
        }
    return results

# Run evaluation
all_results = {}
for prompt, path in test_prompts:
    all_results[prompt] = evaluate_midi(path, prompt)

# Summary table
print(f"\n\n{'='*65}")
print("MELODY TRACK — SUMMARY TABLE")
print(f"{'='*65}")
print(f"{'Prompt':<32} {'Notes':>5} {'Density':>8} {'Interval':>9} {'Entropy':>8}")
print("-"*65)
for prompt, results in all_results.items():
    if "Melody" in results:
        r = results["Melody"]
        print(f"{prompt[:30]:<32} {r['notes']:>5} {r['density']:>8} {r['avg_interval']:>9} {r['pc_entropy']:>8}")

# Training metrics
print(f"\n\n{'='*45}")
print("TRAINING METRICS")
print(f"{'='*45}")
print(f"{'Model':<10} {'Steps':>8} {'Loss':>8} {'Perplexity':>12}")
print("-"*45)
for name, steps, loss in [("Drums",15000,2.3173),("Bass",49000,2.3091),("Chords",50000,1.3857),("Melody",50000,1.5478)]:
    print(f"{name:<10} {steps:>8} {loss:>8.4f} {math.exp(loss):>12.2f}")


Prompt:   'lofi hip hop to chill'
Tempo:    80 BPM | Duration: 24.5s

  [Drums]
  Notes:         82
  Density:       3.35 notes/sec
  Pitch range:   4 semitones
  Avg interval:  1.2 semi  ✅ smooth
  Velocity avg:  51
  PC entropy:    0.60 bits  
  Overlaps:      0  ✅

  [Bass]
  Notes:         51
  Density:       2.08 notes/sec
  Pitch range:   27 semitones
  Avg interval:  7.8 semi  ⚠️ jumpy
  Velocity avg:  55
  PC entropy:    2.41 bits  
  Overlaps:      0  ✅

  [Chords]
  Notes:         69
  Density:       2.82 notes/sec
  Pitch range:   15 semitones
  Avg interval:  3.6 semi  ✅ smooth
  Velocity avg:  55
  PC entropy:    2.61 bits  ✅
  Overlaps:      0  ✅

  [Melody]
  Notes:         61
  Density:       2.49 notes/sec
  Pitch range:   17 semitones
  Avg interval:  2.4 semi  ✅ smooth
  Velocity avg:  55
  PC entropy:    2.48 bits  
  Overlaps:      0  ✅

Prompt:   'epic cinematic soundtrack'
Tempo:    100 BPM | Duration: 20.2s

  [Drums]
  Notes:         51
  Density:       2.53 n

In [ ]:
# Run this for complete per-track metrics
print(f"\n{'='*70}")
print("FULL TRACK ANALYSIS — ALL PROMPTS")
print(f"{'='*70}")

for prompt, path in test_prompts:
    pm = pretty_midi.PrettyMIDI(path)
    _, tempos = pm.get_tempo_changes()
    tempo = tempos[0] if len(tempos) > 0 else 120.0
    print(f"\n📍 '{prompt}' | {tempo:.0f} BPM")
    print(f"   {'Track':<8} {'Notes':>6} {'Density':>8} {'Interval':>9} {'Velocity':>9} {'Overlaps':>9}")
    print(f"   {'-'*55}")
    for inst in pm.instruments:
        if not inst.notes: continue
        notes = inst.notes
        pitches = [n.pitch for n in notes]
        density = len(notes)/pm.get_end_time()
        intervals = [abs(pitches[i+1]-pitches[i]) for i in range(len(pitches)-1)]
        avg_int = np.mean(intervals) if intervals else 0
        avg_vel = np.mean([n.velocity for n in notes])
        starts = sorted([n.start for n in notes])
        gaps = [starts[i+1]-starts[i] for i in range(len(starts)-1)]
        overlaps = sum(1 for g in gaps if g < 0)
        print(f"   {inst.name:<8} {len(notes):>6} {density:>8.2f} {avg_int:>9.2f} {avg_vel:>9.1f} {overlaps:>9}")


FULL TRACK ANALYSIS — ALL PROMPTS

📍 'lofi hip hop to chill' | 80 BPM
   Track     Notes  Density  Interval  Velocity  Overlaps
   -------------------------------------------------------
   Drums        82     3.35      1.19      50.7         0
   Bass         51     2.08      7.78      55.0         0
   Chords       69     2.82      3.65      55.0         0
   Melody       61     2.49      2.43      55.1         0

📍 'epic cinematic soundtrack' | 100 BPM
   Track     Notes  Density  Interval  Velocity  Overlaps
   -------------------------------------------------------
   Drums        51     2.53      2.36      92.1         0
   Bass         61     3.02      2.93     103.0         0
   Chords       54     2.68      5.30     104.8         0
   Melody       47     2.33      2.98     103.0         0

📍 'sad music for a rainy day' | 65 BPM
   Track     Notes  Density  Interval  Velocity  Overlaps
   -------------------------------------------------------
   Drums        92     3.11      